In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, halfnorm, gamma
import seaborn as sns

def A(p):
    return (1 - 2 * p) / (p * (1 - p))

def B(p):
    return 2 / (p * (1 - p))

def C(p, gamma_val):
    return 1 / (int(gamma_val > 0) - p)

def g_gamma(gamma_val):
    """Computes g(gamma) based on the provided formula."""
    return 2 * norm.cdf(-abs(gamma_val)) * np.exp(gamma_val**2 / 2)

def compute_p_gamma(gamma_val, p0):
    """Computes p(gamma; p0) based on the provided formula."""
    indicator = 1 if gamma_val < 0 else 0
    return indicator + (p0 - indicator) / g_gamma(gamma_val)

def sample_Y(mu, sigma, gamma_val, p, n_samples=10000000):
    # Generate random samples based on the given distribution
    epsilon = norm.rvs(size=n_samples)         # N(0, 1)
    z = expon.rvs(scale=1, size=n_samples)     # Exp(1)
    s = halfnorm.rvs(scale=1, size=n_samples)  # N^+(0, 1), half-normal

    term1 = mu
    term2 = C(p, gamma_val) * sigma * abs(gamma_val) * s
    term3 = A(p) * sigma * z
    term4 = np.sqrt(B(p)) * sigma * z * epsilon

    # Calculate samples of Y
    Y = term1 + term2 + term3 + term4
    return Y

def plot_density_overlay(mu_values, sigma_values, gamma_values, p0_values, n_samples=10000000, x_range=(-20, 10)):
    plt.figure(figsize=(12, 8))

    # Use "magma" color palette for a professional look
    colors = sns.color_palette("magma", len(mu_values))

    # Compute actual p values using gamma and p0 values
    p_values = [compute_p_gamma(gamma_val, p0) for gamma_val, p0 in zip(gamma_values, p0_values)]

    for idx, (mu, sigma, gamma_val, p) in enumerate(zip(mu_values, sigma_values, gamma_values, p_values)):
        # Generate samples for the current combination of mu, sigma, gamma, and p
        Y_samples = sample_Y(mu, sigma, gamma_val, p, n_samples=n_samples)

        # Plot density using seaborn for a smooth KDE curve
        sns.kdeplot(Y_samples, bw_adjust=0.5, color=colors[idx], 
                    linestyle='-', linewidth=1.5, alpha=0.7, clip=x_range)

        # Add vertical line at each mu with a lighter color and increased width
        plt.axvline(mu, color=colors[idx], linestyle='--', linewidth=1.5, alpha=0.8)

    # Generate shifted gamma distribution samples and add to the plot
    shape_param = 3   # Gamma shape parameter, can be adjusted for desired skewness
    scale_param = 3.0   # Gamma scale parameter
    gamma_samples = gamma.rvs(a=shape_param, scale=scale_param, size=n_samples) - 17  # Shift by -17

    # Plot shifted gamma density with a teal color
    sns.kdeplot(gamma_samples, bw_adjust=0.5, color='teal', linestyle='-', linewidth=3.5, 
                alpha=0.8, clip=x_range, label="Shifted Gamma Density")

    # Set axis limits
    plt.xlim(x_range)
    plt.ylim(0)  # Start y-axis at 0 for clarity

    # Remove all spines (top, bottom, left, right) for a borderless look
    for spine in plt.gca().spines.values():
        spine.set_visible(False)

    # Remove x-axis and y-axis labels and ticks for a clean look
    plt.gca().set_xticks([])
    plt.gca().set_yticks([])
    plt.gca().set_xlabel('')
    plt.gca().set_ylabel('')

    # Save plot as a high-quality PNG image
    plt.savefig("density_plot.png", dpi=300, bbox_inches='tight', pad_inches=0)

    # Show plot
    plt.show()

# Set parameters
mu_values = [-15, -13, -12, -9, -5, 0, 6]
sigma_values = [0.256, 0.556, 0.508, 0.870, 0.849, 0.799, 0.591]
gamma_values = [0.343 , -0.0181, -0.108 , -0.185, -0.411, -0.905, -2.57]
p0_values = [0.05, 0.2, 0.35, 0.5, 0.65, 0.8, 0.95]

# Plot densities with specified x-axis range
plot_density_overlay(mu_values, sigma_values, gamma_values, p0_values, x_range=(-18, 8))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, halfnorm, gamma
import seaborn as sns

def A(p):
    return (1 - 2 * p) / (p * (1 - p))

def B(p):
    return 2 / (p * (1 - p))

def C(p, gamma_val):
    return 1 / (int(gamma_val > 0) - p)

def g_gamma(gamma_val):
    """Computes g(gamma) based on the provided formula."""
    return 2 * norm.cdf(-abs(gamma_val)) * np.exp(gamma_val**2 / 2)

def compute_p_gamma(gamma_val, p0):
    """Computes p(gamma; p0) based on the provided formula."""
    indicator = 1 if gamma_val < 0 else 0
    return indicator + (p0 - indicator) / g_gamma(gamma_val)

def sample_Y(mu, sigma, gamma_val, p, n_samples=1000000):
    # Generate random samples based on the given distribution
    epsilon = norm.rvs(size=n_samples)         # N(0, 1)
    z = expon.rvs(scale=1, size=n_samples)     # Exp(1)
    s = halfnorm.rvs(scale=1, size=n_samples)  # N^+(0, 1), half-normal

    term1 = mu
    term2 = C(p, gamma_val) * sigma * abs(gamma_val) * s
    term3 = A(p) * sigma * z
    term4 = np.sqrt(B(p)) * sigma * z * epsilon

    # Calculate samples of Y
    Y = term1 + term2 + term3 + term4
    return Y

def plot_cdf_overlay(mu_values, sigma_values, gamma_values, p0_values, n_samples=1000000, x_range=(-20, 10)):
    plt.figure(figsize=(12, 8))

    # Use "magma" color palette for a professional look
    colors = sns.color_palette("magma", len(mu_values))

    # Compute actual p values using gamma and p0 values
    p_values = [compute_p_gamma(gamma_val, p0) for gamma_val, p0 in zip(gamma_values, p0_values)]

    for idx, (mu, sigma, gamma_val, p) in enumerate(zip(mu_values, sigma_values, gamma_values, p_values)):
        # Generate samples for the current combination of mu, sigma, gamma, and p
        Y_samples = sample_Y(mu, sigma, gamma_val, p, n_samples=n_samples)
        
        # Sort the samples for CDF plotting
        Y_sorted = np.sort(Y_samples)
        cdf = np.linspace(0, 1, len(Y_sorted))

        # Plot the CDF
        plt.plot(Y_sorted, cdf, color=colors[idx], linestyle='-', linewidth=1.5, alpha=0.7, label=f'Mu: {mu}, Sigma: {sigma}')

        # Add vertical line at each mu
        # plt.axvline(mu, color=colors[idx], linestyle='--', linewidth=1.5, alpha=0.8)

    # Generate shifted gamma distribution samples and add to the plot
    shape_param = 3   # Gamma shape parameter, can be adjusted for desired skewness
    scale_param = 3.0   # Gamma scale parameter
    gamma_samples = gamma.rvs(a=shape_param, scale=scale_param, size=n_samples) - 17  # Shift by -17
    gamma_sorted = np.sort(gamma_samples)
    gamma_cdf = np.linspace(0, 1, len(gamma_sorted))

    # Plot shifted gamma CDF with a teal color
    plt.plot(gamma_sorted, gamma_cdf, color='teal', linestyle='-', linewidth=3.5, alpha=0.8, label="Shifted Gamma CDF")

    # Set axis limits
    plt.xlim(x_range)
    plt.ylim(0, 1)  # CDF always ranges between 0 and 1

    # Add grid and legend
    plt.grid(True, linestyle='--', alpha=0.5)
    # plt.legend(fontsize=10)

    # Customize labels
    plt.title('Comparison of CDFs Across Distributions', fontsize=14)
    plt.xlabel('Y', fontsize=12)
    plt.ylabel('CDF', fontsize=12)

    # Save plot as a high-quality PNG image
    plt.savefig("cdf_plot.png", dpi=300, bbox_inches='tight', pad_inches=0)

    # Show plot
    plt.show()

# Set parameters
mu_values = [-15, -13, -12, -9, -5, 0, 6]
sigma_values = [0.256, 0.556, 0.508, 0.870, 0.849, 0.799, 0.591]
gamma_values = [0.343 , -0.0181, -0.108 , -0.185, -0.411, -0.905, -2.57]
p0_values = [0.05, 0.2, 0.35, 0.5, 0.65, 0.8, 0.95]

# Plot CDFs with specified x-axis range
plot_cdf_overlay(mu_values, sigma_values, gamma_values, p0_values, x_range=(-18, 8))
